In [7]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import pandas as pd
import os
from datetime import datetime
import json

class LunchMenuRecommender:
    def __init__(self, root):
        self.root = root
        self.root.title("오늘의 점심 메뉴")
        self.root.geometry("600x700")
        self.root.configure(bg="#f5f5f5")
        
        # 앱 데이터
        self.restaurant_data = self.load_restaurant_data()
        self.food_categories = sorted(list(set(restaurant["카테고리"] for restaurant in self.restaurant_data)))
        self.price_ranges = ["전체", "~8,000원", "~12,000원", "~15,000원", "15,000원 이상"]
        self.distances = ["전체", "5분 이내", "10분 이내", "15분 이내", "20분 이내"]
        
        # 사용자 기록
        self.history_file = "lunch_history.json"
        self.user_history = self.load_history()
        
        # 추천 결과
        self.current_recommendations = []
        
        # UI 구성
        self.setup_ui()
        
    def load_restaurant_data(self):
        """내장된 식당 데이터 또는 파일에서 데이터 로드"""
        # 우선 기본 데이터로 시작
        default_data = [
            {"이름": "행복 김밥", "카테고리": "분식", "메뉴": "김밥, 라면", "가격대": "7,000원", "위치": "회사 앞 1분", "특이사항": "점심 특선 세트 있음"},
            {"이름": "맛있는 국수", "카테고리": "한식", "메뉴": "국수, 비빔밥", "가격대": "8,000원", "위치": "회사 왼쪽 3분", "특이사항": "김치 무한리필"},
            {"이름": "웍앤박스", "카테고리": "중식", "메뉴": "짜장면, 짬뽕", "가격대": "9,000원", "위치": "회사 오른쪽 5분", "특이사항": "탕수육 세트 추천"},
            {"이름": "파스타 하우스", "카테고리": "양식", "메뉴": "파스타, 리조또", "가격대": "14,000원", "위치": "회사 뒤 7분", "특이사항": "와인 한 잔 서비스"},
            {"이름": "스시 히로", "카테고리": "일식", "메뉴": "초밥, 우동", "가격대": "16,000원", "위치": "회사 정문 10분", "특이사항": "점심 특선 세트 있음"},
            {"이름": "왕돈까스", "카테고리": "일식", "메뉴": "돈까스, 카레", "가격대": "11,000원", "위치": "회사 후문 8분", "특이사항": "사이드 메뉴 풍부"},
            {"이름": "소담 정식", "카테고리": "한식", "메뉴": "된장찌개, 제육볶음", "가격대": "9,000원", "위치": "회사 정문 3분", "특이사항": "반찬 종류 많음"},
            {"이름": "타이 익스프레스", "카테고리": "아시안", "메뉴": "팟타이, 카오팟", "가격대": "10,000원", "위치": "회사 왼쪽 12분", "특이사항": "매운맛 조절 가능"},
            {"이름": "멕시칸 그릴", "카테고리": "멕시칸", "메뉴": "타코, 부리또", "가격대": "13,000원", "위치": "회사 오른쪽 15분", "특이사항": "할라피뇨 추가 가능"},
            {"이름": "샐러드 바", "카테고리": "샐러드", "메뉴": "각종 샐러드", "가격대": "12,000원", "위치": "회사 정문 2분", "특이사항": "토핑 추가 가능"},
            {"이름": "라멘 하우스", "카테고리": "일식", "메뉴": "라멘, 규동", "가격대": "11,000원", "위치": "회사 후문 6분", "특이사항": "숙주 무한리필"},
            {"이름": "빕스버거", "카테고리": "패스트푸드", "메뉴": "햄버거, 치킨버거", "가격대": "8,000원", "위치": "회사 정문 4분", "특이사항": "세트 주문시 음료 업그레이드"},
            {"이름": "베트남 쌀국수", "카테고리": "아시안", "메뉴": "쌀국수, 월남쌈", "가격대": "10,000원", "위치": "회사 왼쪽 8분", "특이사항": "소고기 추가 가능"},
            {"이름": "안녕 돈부리", "카테고리": "일식", "메뉴": "가츠동, 규동", "가격대": "12,000원", "위치": "회사 오른쪽 3분", "특이사항": "밥 리필 가능"},
            {"이름": "마마 떡볶이", "카테고리": "분식", "메뉴": "떡볶이, 순대", "가격대": "7,000원", "위치": "회사 뒤 4분", "특이사항": "매운맛 단계 조절 가능"},
            {"이름": "북촌손만두", "카테고리": "한식", "메뉴": "만두, 칼국수", "가격대": "9,000원", "위치": "회사 정문 7분", "특이사항": "주말에는 혼잡함"},
            {"이름": "스테이크 하우스", "카테고리": "양식", "메뉴": "스테이크, 필라프", "가격대": "17,000원", "위치": "회사 후문 12분", "특이사항": "점심 특선 메뉴 있음"},
            {"이름": "청년 피자", "카테고리": "양식", "메뉴": "피자, 파스타", "가격대": "13,000원", "위치": "회사 왼쪽 6분", "특이사항": "2인 세트 메뉴 있음"},
            {"이름": "사철 갈비탕", "카테고리": "한식", "메뉴": "갈비탕, 설렁탕", "가격대": "12,000원", "위치": "회사 오른쪽 9분", "특이사항": "고기 양 많음"},
            {"이름": "베이글 카페", "카테고리": "브런치", "메뉴": "베이글, 샌드위치", "가격대": "8,000원", "위치": "회사 정문 5분", "특이사항": "아메리카노 1+1"}
        ]
        
        # 엑셀 파일이 있으면 로드 시도
        excel_path = "restaurant_list.xlsx"
        if os.path.exists(excel_path):
            try:
                df = pd.read_excel(excel_path)
                # 데이터프레임을 딕셔너리 리스트로 변환
                return df.to_dict('records')
            except Exception as e:
                print(f"엑셀 파일 로드 실패: {e}")
        
        # 엑셀 파일 없으면 기본 데이터 반환
        return default_data
    
    def load_history(self):
        """사용자의 선택 기록 로드"""
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except:
                return {"selected": [], "rejected": [], "last_visit_dates": {}}
        return {"selected": [], "rejected": [], "last_visit_dates": {}}
    
    def save_history(self):
        """사용자 기록 저장"""
        with open(self.history_file, 'w', encoding='utf-8') as f:
            json.dump(self.user_history, f, ensure_ascii=False, indent=2)
    
    def setup_ui(self):
        """UI 구성 요소 설정"""
        # 메인 프레임
        self.main_frame = tk.Frame(self.root, bg="#f5f5f5")
        self.main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)
        
        # 앱 제목
        title_frame = tk.Frame(self.main_frame, bg="#f5f5f5")
        title_frame.pack(fill=tk.X, pady=(0, 20))
        
        title_label = tk.Label(title_frame, text="오늘의 점심 메뉴", 
                              font=("맑은 고딕", 24, "bold"), fg="#333333", bg="#f5f5f5")
        title_label.pack()
        
        subtitle_label = tk.Label(title_frame, text="더 이상 고민하지 마세요!",
                                font=("맑은 고딕", 12), fg="#666666", bg="#f5f5f5")
        subtitle_label.pack(pady=(5, 0))
        
        # 필터 프레임
        filter_frame = tk.LabelFrame(self.main_frame, text="메뉴 필터", 
                                   font=("맑은 고딕", 11, "bold"), bg="#f5f5f5")
        filter_frame.pack(fill=tk.X, pady=(0, 20), ipady=10, ipadx=10)
        
        # 음식 카테고리 선택
        category_frame = tk.Frame(filter_frame, bg="#f5f5f5")
        category_frame.pack(fill=tk.X, pady=5)
        
        tk.Label(category_frame, text="음식 종류:", font=("맑은 고딕", 11), bg="#f5f5f5").pack(side=tk.LEFT, padx=(10, 5))
        
        self.category_var = tk.StringVar(value="전체")
        category_options = ["전체"] + self.food_categories
        self.category_combo = ttk.Combobox(category_frame, values=category_options, textvariable=self.category_var, width=15, state="readonly")
        self.category_combo.pack(side=tk.LEFT, padx=(0, 10))
        
        # 가격대 선택
        price_frame = tk.Frame(filter_frame, bg="#f5f5f5")
        price_frame.pack(fill=tk.X, pady=5)
        
        tk.Label(price_frame, text="가격대:", font=("맑은 고딕", 11), bg="#f5f5f5").pack(side=tk.LEFT, padx=(10, 5))
        
        self.price_var = tk.StringVar(value="전체")
        self.price_combo = ttk.Combobox(price_frame, values=self.price_ranges, textvariable=self.price_var, width=15, state="readonly")
        self.price_combo.pack(side=tk.LEFT, padx=(0, 10))
        
        # 거리 선택
        distance_frame = tk.Frame(filter_frame, bg="#f5f5f5")
        distance_frame.pack(fill=tk.X, pady=5)
        
        tk.Label(distance_frame, text="거리:", font=("맑은 고딕", 11), bg="#f5f5f5").pack(side=tk.LEFT, padx=(10, 5))
        
        self.distance_var = tk.StringVar(value="전체")
        self.distance_combo = ttk.Combobox(distance_frame, values=self.distances, textvariable=self.distance_var, width=15, state="readonly")
        self.distance_combo.pack(side=tk.LEFT, padx=(0, 10))
        
        # 최근 방문 제외 옵션
        exclude_frame = tk.Frame(filter_frame, bg="#f5f5f5")
        exclude_frame.pack(fill=tk.X, pady=5)
        
        self.exclude_recent_var = tk.BooleanVar(value=True)
        exclude_check = tk.Checkbutton(exclude_frame, text="최근 3일 내 방문한 곳 제외", variable=self.exclude_recent_var, 
                                      font=("맑은 고딕", 11), bg="#f5f5f5", activebackground="#f5f5f5")
        exclude_check.pack(side=tk.LEFT, padx=10)
        
        # 버튼 프레임
        button_frame = tk.Frame(self.main_frame, bg="#f5f5f5")
        button_frame.pack(fill=tk.X, pady=(0, 20))
        
        self.recommend_btn = tk.Button(
            button_frame, 
            text="메뉴 추천받기", 
            command=self.get_recommendations,
            font=("맑은 고딕", 12, "bold"),
            bg="#4CAF50",
            fg="white",
            activebackground="#45a049",
            activeforeground="white",
            height=2,
            cursor="hand2"
        )
        self.recommend_btn.pack(fill=tk.X)
        
        # 결과 프레임
        self.result_frame = tk.LabelFrame(self.main_frame, text="추천 결과", 
                                        font=("맑은 고딕", 11, "bold"), bg="#f5f5f5")
        self.result_frame.pack(fill=tk.BOTH, expand=True, ipady=10, ipadx=10)
        
        self.recommendation_label = tk.Label(
            self.result_frame,
            text="아래 버튼을 클릭하여 추천을 받아보세요!",
            font=("맑은 고딕", 12),
            bg="#f5f5f5",
            wraplength=500
        )
        self.recommendation_label.pack(pady=(20, 0))
        
        # 결과 카드 프레임
        self.card_frame = tk.Frame(self.result_frame, bg="#f5f5f5")
        self.card_frame.pack(fill=tk.BOTH, expand=True, pady=10)
        
        # 하단 버튼 프레임
        self.bottom_frame = tk.Frame(self.main_frame, bg="#f5f5f5")
        self.bottom_frame.pack(fill=tk.X, pady=(10, 0))
        
        # 이전 기록 버튼
        self.history_btn = tk.Button(
            self.bottom_frame,
            text="최근 선택 기록",
            command=self.show_history,
            font=("맑은 고딕", 10),
            bg="#e0e0e0",
            cursor="hand2"
        )
        self.history_btn.pack(side=tk.LEFT)
        
        # 엑셀 파일 생성/수정 버튼
        self.excel_btn = tk.Button(
            self.bottom_frame,
            text="식당 목록 관리",
            command=self.manage_restaurant_list,
            font=("맑은 고딕", 10),
            bg="#e0e0e0",
            cursor="hand2"
        )
        self.excel_btn.pack(side=tk.RIGHT)
        
        # 스타일 설정
        self.setup_styles()
        
    def setup_styles(self):
        """위젯 스타일 설정"""
        style = ttk.Style()
        style.configure("TCombobox", fieldbackground="#ffffff", background="#ffffff")
        
        # 버튼에 마우스 오버 효과 추가
        self.recommend_btn.bind("<Enter>", lambda e: self.recommend_btn.config(bg="#45a049"))
        self.recommend_btn.bind("<Leave>", lambda e: self.recommend_btn.config(bg="#4CAF50"))
        
    def get_recommendations(self):
        """필터링된 식당 목록에서 추천 받기"""
        filtered_restaurants = self.filter_restaurants()
        
        if not filtered_restaurants:
            messagebox.showinfo("알림", "조건에 맞는 식당이 없습니다.\n필터 조건을 변경해보세요.")
            return
        
        # 최대 3개 식당 랜덤 추천
        num_to_recommend = min(3, len(filtered_restaurants))
        self.current_recommendations = random.sample(filtered_restaurants, num_to_recommend)
        
        self.display_recommendations()
    
    def filter_restaurants(self):
        """필터 조건에 맞는 식당 필터링"""
        filtered = self.restaurant_data.copy()
        
        # 카테고리 필터
        if self.category_var.get() != "전체":
            filtered = [r for r in filtered if r["카테고리"] == self.category_var.get()]
        
        # 가격대 필터
        if self.price_var.get() != "전체":
            max_price = int(self.price_var.get().replace("~", "").replace(",", "").replace("원", ""))
            filtered = [r for r in filtered if self.extract_price(r["가격대"]) <= max_price]
        
        # 거리 필터
        if self.distance_var.get() != "전체":
            max_minutes = int(self.distance_var.get().replace("분 이내", ""))
            filtered = [r for r in filtered if self.extract_minutes(r["위치"]) <= max_minutes]
        
        # 최근 방문 제외
        if self.exclude_recent_var.get():
            recent_restaurants = []
            today = datetime.now().date()
            
            for name, date_str in self.user_history["last_visit_dates"].items():
                if date_str:
                    visit_date = datetime.strptime(date_str, "%Y-%m-%d").date()
                    days_since_visit = (today - visit_date).days
                    if days_since_visit <= 3:
                        recent_restaurants.append(name)
            
            filtered = [r for r in filtered if r["이름"] not in recent_restaurants]
        
        return filtered
    
    def extract_price(self, price_str):
        """가격 문자열에서 숫자만 추출"""
        import re
        numbers = re.findall(r'\d+', price_str.replace(",", ""))
        if numbers:
            return int(numbers[0])
        return 999999  # 가격을 추출할 수 없는 경우 매우 큰 값 반환
    
    def extract_minutes(self, location_str):
        """위치 문자열에서 분 단위 시간 추출"""
        import re
        numbers = re.findall(r'\d+', location_str)
        if numbers:
            return int(numbers[0])
        return 999  # 시간을 추출할 수 없는 경우 매우 큰 값 반환
    
    def display_recommendations(self):
        """추천 결과 표시"""
        # 기존 카드 삭제
        for widget in self.card_frame.winfo_children():
            widget.destroy()
        
        if not self.current_recommendations:
            self.recommendation_label.config(text="조건에 맞는 식당이 없습니다.\n필터 조건을 변경해보세요.")
            return
        
        self.recommendation_label.config(text="오늘의 추천 메뉴입니다!")
        
        # 추천 카드 생성
        for idx, restaurant in enumerate(self.current_recommendations):
            card = tk.Frame(self.card_frame, bg="white", bd=1, relief=tk.RAISED)
            card.pack(fill=tk.X, pady=5, padx=10, ipady=5)
            
            # 식당 이름과 카테고리
            header = tk.Frame(card, bg="white")
            header.pack(fill=tk.X, padx=10, pady=(5,0))
            
            name_label = tk.Label(header, text=restaurant["이름"], font=("맑은 고딕", 12, "bold"), fg="#333333", bg="white")
            name_label.pack(side=tk.LEFT)
            
            category_label = tk.Label(header, text=f"[{restaurant['카테고리']}]", font=("맑은 고딕", 10), fg="#666666", bg="white")
            category_label.pack(side=tk.LEFT, padx=(5, 0))
            
            # 메뉴, 가격, 위치, 특이사항
            info_frame = tk.Frame(card, bg="white")
            info_frame.pack(fill=tk.X, padx=10, pady=(3, 5))
            
            menu_label = tk.Label(info_frame, text=f"메뉴: {restaurant['메뉴']}", font=("맑은 고딕", 10), fg="#333333", bg="white", anchor="w")
            menu_label.pack(fill=tk.X, pady=1)
            
            price_label = tk.Label(info_frame, text=f"가격대: {restaurant['가격대']}", font=("맑은 고딕", 10), fg="#333333", bg="white", anchor="w")
            price_label.pack(fill=tk.X, pady=1)
            
            location_label = tk.Label(info_frame, text=f"위치: {restaurant['위치']}", font=("맑은 고딕", 10), fg="#333333", bg="white", anchor="w")
            location_label.pack(fill=tk.X, pady=1)
            
            if restaurant["특이사항"]:
                note_label = tk.Label(info_frame, text=f"특이사항: {restaurant['특이사항']}", font=("맑은 고딕", 10), fg="#666666", bg="white", anchor="w")
                note_label.pack(fill=tk.X, pady=1)
            
            # 버튼 프레임
            button_frame = tk.Frame(card, bg="white")
            button_frame.pack(fill=tk.X, padx=10, pady=(5, 5))
            
            # 선택 버튼
            select_btn = tk.Button(
                button_frame,
                text="✓ 이걸로 갈래요!",
                command=lambda r=restaurant: self.select_restaurant(r),
                bg="#4CAF50",
                fg="white",
                font=("맑은 고딕", 10),
                cursor="hand2"
            )
            select_btn.pack(side=tk.LEFT, padx=(0, 5))
            
            # 재추천 버튼
            if idx == 0:  # 첫 번째 카드에만 재추천 버튼 추가
                reroll_btn = tk.Button(
                    button_frame,
                    text="↻ 다시 추천받기",
                    command=self.get_recommendations,
                    bg="#FF9800",
                    fg="white",
                    font=("맑은 고딕", 10),
                    cursor="hand2"
                )
                reroll_btn.pack(side=tk.RIGHT)
    
    def select_restaurant(self, restaurant):
        """식당 선택 처리"""
        today_str = datetime.now().date().isoformat()
        
        # 방문 기록 업데이트
        self.user_history["selected"].append({
            "name": restaurant["이름"],
            "date": today_str,
            "category": restaurant["카테고리"]
        })
        
        # 최대 30개 기록만 유지
        if len(self.user_history["selected"]) > 30:
            self.user_history["selected"].pop(0)
        
        # 마지막 방문일 업데이트
        self.user_history["last_visit_dates"][restaurant["이름"]] = today_str
        
        # 기록 저장
        self.save_history()
        
        # 메시지 표시
        messagebox.showinfo("선택 완료", f"오늘의 점심은 '{restaurant['이름']}'으로 결정되었습니다!\n맛있게 드세요!")
        
        # UI 업데이트
        self.recommendation_label.config(text=f"'{restaurant['이름']}'을(를) 선택했습니다. 맛있게 드세요!")
        for widget in self.card_frame.winfo_children():
            widget.destroy()
    
    def show_history(self):
        """최근 선택 기록 표시"""
        if not self.user_history["selected"]:
            messagebox.showinfo("기록 없음", "아직 선택 기록이 없습니다.")
            return
        
        history_window = tk.Toplevel(self.root)
        history_window.title("최근 선택 기록")
        history_window.geometry("400x500")
        history_window.configure(bg="#f5f5f5")
        
        # 제목
        tk.Label(
            history_window, 
            text="최근 방문 기록", 
            font=("맑은 고딕", 16, "bold"),
            bg="#f5f5f5"
        ).pack(pady=(15, 10))
        
        # 기록 표시 프레임
        history_frame = tk.Frame(history_window, bg="#f5f5f5")
        history_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=10)
        
        # 스크롤바
        scrollbar = tk.Scrollbar(history_frame)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # 리스트박스
        history_list = tk.Listbox(
            history_frame,
            font=("맑은 고딕", 11),
            height=20,
            width=40,
            bd=1,
            selectbackground="#a6a6a6",
            yscrollcommand=scrollbar.set
        )
        history_list.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar.config(command=history_list.yview)
        
        # 기록 항목 추가 (최근 순으로)
        for item in reversed(self.user_history["selected"]):
            date_obj = datetime.strptime(item["date"], "%Y-%m-%d").date()
            formatted_date = date_obj.strftime("%Y년 %m월 %d일")
            history_list.insert(tk.END, f"{formatted_date}: {item['name']} ({item['category']})")
        
        # 닫기 버튼
        tk.Button(
            history_window,
            text="닫기",
            command=history_window.destroy,
            font=("맑은 고딕", 11),
            bg="#e0e0e0",
            cursor="hand2"
        ).pack(pady=(0, 15))
    
    def manage_restaurant_list(self):
        """식당 목록 관리 (엑셀 파일 생성 및 수정)"""
        # 현재 데이터로 엑셀 파일 생성
        df = pd.DataFrame(self.restaurant_data)
        excel_path = "restaurant_list.xlsx"
        
        try:
            df.to_excel(excel_path, index=False)
            messagebox.showinfo(
                "식당 목록 관리", 
                f"식당 목록이 '{excel_path}' 파일로 생성되었습니다.\n\n"
                "이 파일을 수정한 후 프로그램을 다시 실행하면 변경사항이 반영됩니다."
            )
        except Exception as e:
            messagebox.showerror("오류", f"파일 생성 중 오류가 발생했습니다.\n{str(e)}")

# 앱 실행
if __name__ == "__main__":
    root = tk.Tk()
    app = LunchMenuRecommender(root)
    root.mainloop()
